In [1]:
# ============================================
# STEP 1: Setup — Imports, Paths, Configuration
# ============================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

RANDOM_STATE = 42

PROJECT_ROOT = Path("../")
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Report: T2-A - Identifier Permutation Robustness")
print("Models: Decision Tree, LightGBM (Set A and Set B Models)")
print("Random state:", RANDOM_STATE)

Report: T2-A - Identifier Permutation Robustness
Models: Decision Tree, LightGBM (Set A and Set B Models)
Random state: 42


## T2-A: Identifier Permutation Robustness

### Hypothesis H2

Set A (identifier-inclusive) models will show a **significant Macro-F1 drop** when the identifier columns (src_ip, src_port, dst_ip, dst_port) in the test set are randomly permuted (shuffled independently, breaking any real correlation with the attack label), compared to the clean T1 holdout result — because this would prove the model learned spurious identifier-label associations (e.g. "src_ip=192.168.1.31 → always attack") rather than genuine behavioural patterns.

### Protocol

- **Starting point:** the same locked T1 test set (same rows, same `random_state=42` split)
- **Perturbation:** `src_ip`, `src_port`, `dst_ip`, `dst_port` columns are each independently shuffled (`np.random.permutation`) within the test set — this breaks the row-wise pairing between identifier values and their original attack label, while keeping the same overall identifier value distribution
- **Models:** Decision Tree (Set A) and LightGBM (Set A) — reused from `04a`/`04d`, no retraining
- **Comparison:** T2-A Macro-F1 vs T1 Macro-F1 (same model, same feature set, only the identifier columns changed)
- **Expected result if H2 holds:** a large drop for Set A models (since they can no longer rely on memorized identifier-label shortcuts); Set B is not tested here since it has no identifier columns to permute

In [2]:
# ============================================
# STEP 3: Recreate Test Set + Apply Unified 5-Identifier Permutation
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

FEATURE_SETS_PATH = PROJECT_ROOT / "data" / "processed" / "feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

all_features = feature_sets["set_a_columns"]
set_b_features = feature_sets["set_b_columns"]
# All 5 excluded identifiers as documented in threat model
identifier_cols = feature_sets["excluded_identifiers"]

SET_A_PATH = PROJECT_ROOT / "data" / "processed" / "set_a_identifier_inclusive.csv"
df = pd.read_csv(SET_A_PATH)

X = df[all_features].copy()
y = df["type"].copy()

# Leakage-free split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

X_test_raw = X_test.copy()
X_test_filled = X_test.fillna(-1)

print("Clean test set (locked):", X_test_raw.shape[0], "rows")

# ---- Apply T2-A permutation: synchronize draw across raw and filled matrices ----
rng = np.random.default_rng(RANDOM_STATE)
X_test_raw_perm = X_test_raw.copy()
X_test_filled_perm = X_test_filled.copy()

for col in identifier_cols:
    perm_vals = rng.permutation(X_test_raw[col].values)
    X_test_raw_perm[col] = perm_vals
    X_test_filled_perm[col] = perm_vals  # EXACT SAME values assigned

print("\nAll 5 identifier columns permuted:", identifier_cols)
# Assert that non-identifier (Set B) columns are 100% untouched
assert X_test_raw_perm[set_b_features].equals(X_test_raw[set_b_features])
assert X_test_filled_perm[set_b_features].equals(X_test_filled[set_b_features])
print("Sanity check passed: Set B features are 100% bit-for-bit identical.")

Clean test set (locked): 28572 rows

All 5 identifier columns permuted: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'dns_query_has_ip_pattern']
Sanity check passed: Set B features are 100% bit-for-bit identical.


C:\Users\User\AppData\Local\Temp\ipykernel_18820\89931610.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


In [3]:
# ============================================
# STEP 4: Evaluate T2-A Robustness for Both Set A and Set B
# ============================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_files_t2a = {
    ("Decision Tree", "Set A"): ("dt_set_a.joblib", X_test_raw, X_test_raw_perm, all_features),
    ("Decision Tree", "Set B"): ("dt_set_b.joblib", X_test_raw, X_test_raw_perm, set_b_features),
    ("LightGBM", "Set A"): ("lgbm_set_a.joblib", X_test_filled, X_test_filled_perm, all_features),
    ("LightGBM", "Set B"): ("lgbm_set_b.joblib", X_test_filled, X_test_filled_perm, set_b_features),
}

t2a_results = []

for (model_name, feat_set), (filename, clean_set, perm_set, features) in model_files_t2a.items():
    model = joblib.load(MODELS_DIR / filename)
    clean_preds = model.predict(clean_set[features])
    perm_preds = model.predict(perm_set[features])

    clean_f1 = f1_score(y_test, clean_preds, average="macro", zero_division=0)
    perm_f1 = f1_score(y_test, perm_preds, average="macro", zero_division=0)

    # Verify Set B invariance assertion
    if feat_set == "Set B":
        matches = (clean_preds == perm_preds).sum()
        assert np.array_equal(clean_preds, perm_preds), f"Set B predictions changed: {matches}/{len(clean_preds)}"
        print(f"[Verified] {model_name} Set B prediction equality: 100% ({matches}/{len(clean_preds)})")

    t2a_results.append({
        "Model": model_name,
        "Feature Set": feat_set,
        "T1 Macro-F1 (clean)": clean_f1,
        "T2-A Macro-F1 (permuted)": perm_f1,
        "Drop": round(clean_f1 - perm_f1, 4),
        "Accuracy": accuracy_score(y_test, perm_preds),
        "Precision": precision_score(y_test, perm_preds, average="macro", zero_division=0),
        "Recall": recall_score(y_test, perm_preds, average="macro", zero_division=0),
    })

t2a_df = pd.DataFrame(t2a_results)
print("\n=== T2-A Identifier Permutation Results ===")
print(t2a_df.round(4).to_string(index=False))

t2a_df.to_csv(RESULTS_DIR / "t2a_permutation_results.csv", index=False)
print("\nSaved to:", RESULTS_DIR / "t2a_permutation_results.csv")


[Verified] Decision Tree Set B prediction equality: 100% (28572/28572)
[Verified] LightGBM Set B prediction equality: 100% (28572/28572)

=== T2-A Identifier Permutation Results ===
        Model Feature Set  T1 Macro-F1 (clean)  T2-A Macro-F1 (permuted)   Drop  Accuracy  Precision  Recall
Decision Tree       Set A               0.9806                    0.2252 0.7553    0.2672     0.2615  0.2241
Decision Tree       Set B               0.9471                    0.9471 0.0000    0.9761     0.9472  0.9472
     LightGBM       Set A               0.9861                    0.5380 0.4481    0.5842     0.5667  0.5376
     LightGBM       Set B               0.8756                    0.8756 0.0000    0.9237     0.8758  0.8912

Saved to: ..\results\t2a_permutation_results.csv


## T2-A Result & H2 Conclusion

### Canonical Results (Seed 42)
| Model | Feature Set | T1 Clean Macro-F1 | T2-A Permuted Macro-F1 | Drop (Δ) |
|---|:---:|:---:|:---:|:---:|
| Decision Tree | Set A (Inclusive) | 0.9806 | 0.2252 | **0.7553** (Collapse) |
| Decision Tree | **Set B (Behavioral)** | **0.9471** | **0.9471** | **0.0000** (Invariant) |
| LightGBM | Set A (Inclusive) | 0.9861 | 0.5380 | **0.4481** (Collapse) |
| LightGBM | **Set B (Behavioral)** | **0.8756** | **0.8756** | **0.0000** (Invariant) |

**Hypothesis H2: Strongly Accepted.** When network identifiers are randomly permuted, Set A models suffer catastrophic collapse (DT: −75.5 pp, LightGBM: −44.8 pp) because their learned decision paths relied on spurious identifier-label shortcuts. In contrast, Set B (behavioural-only) models demonstrate complete **identifier invariance by construction** — verified via bit-for-bit prediction equality (28,572/28,572 matches for both models) — confirming they cannot be manipulated through identifier spoofing.

##REMOVE BELOW

## T2-A Result & H2 Conclusion

### Canonical Results (Seed 42)
| Model | Feature Set | T1 Clean Macro-F1 | T2-A Permuted Macro-F1 | Drop (Δ) |
|---|:---:|:---:|:---:|:---:|
| Decision Tree | Set A (Inclusive) | 0.9810 | 0.2237 | **0.7573** (Collapse) |
| Decision Tree | **Set B (Behavioral)** | **0.9471** | **0.9471** | **0.0000** (Invariant) |
| LightGBM | Set A (Inclusive) | 0.9861 | 0.5476 | **0.4385** (Collapse) |
| LightGBM | **Set B (Behavioral)** | **0.8756** | **0.8756** | **0.0000** (Invariant) |

### Multi-Seed Statistical Summary (5 Seeds: Mean ± Std)
- **Decision Tree Set A:** Clean $0.9821 \pm 0.0012 \rightarrow$ Permuted $0.2446 \pm 0.0109$ (Drop: $0.7375$).
- **LightGBM Set A:** Clean $0.9872 \pm 0.0018 \rightarrow$ Permuted $0.5624 \pm 0.0156$ (Drop: $0.4248$).
- **Set B (Both Models):** Identical predictions in $100\%$ of evaluations across all seeds ($285,720 / 285,720$ matches, $\Delta = \mathbf{0.000000}$).

**Hypothesis H2: Strongly Accepted.** When network identifiers are randomized, Set A models suffer catastrophic collapse because their learned decision paths relied on spurious shortcuts. In contrast, Set B (behavioural-only) models demonstrate complete **identifier invariance by construction**, outperforming Set A under attack by **+70.3 percentage points (Decision Tree)** and **+31.9 percentage points (LightGBM)**.